# 🧠 Planner Model — Google Colab Notebook
## Retail Analytics Copilot — `LOCAL_MODELS=False`

هذا الـ notebook يشغّل موديل **Planner** على Google Colab ويكشفه عبر Ngrok.

**دور الـ Planner:** يستخرج القيود المنظّمة من السؤال:
- نطاق التواريخ (`date_range`): مثل "يونيو 1997" → `{start: '1997-06-01', end: '1997-06-30'}`
- صيغة KPI (`kpi_formula`): مثل "هامش الربح" → `SUM(UnitPrice*0.3*Quantity*(1-Discount))`
- الفئات (`categories`): مثل "مشروبات" → `["Beverages"]`
- الكيانات (`entities`): أسماء حملات تسويقية أو مواسم

**الموديل المستخدم:** `phi3.5:3.8b-mini-instruct-q4_K_M` (سريع ومناسب لاستخراج JSON)

---
### ⚠️ تعليمات ما قبل التشغيل:
1. تأكد من تفعيل **GPU Runtime** → Runtime > Change runtime type > T4 GPU
2. ضع `NGROK_AUTH_TOKEN` في Colab Secrets
3. بعد التشغيل، انسخ رابط Ngrok وضعه في `.env` → `NGROK_PLANNER_URL`

In [ ]:
# ─── الخلية 1: إيقاف أي Ollama قديم وتحديث النظام ───────────────────────────
# تنظيف البيئة وتثبيت المتطلبات الأساسية
!pkill ollama || true
!apt-get update -qq
!apt-get install -y -qq zstd

In [ ]:
# ─── الخلية 2: تثبيت Ollama ────────────────────────────────────────────────
# تثبيت Ollama runtime لتشغيل نماذج GGUF
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
# ─── الخلية 3: تشغيل خادم Ollama على المنفذ 8000 ─────────────────────────────
# الـ Planner يُستدعى في بداية كل سؤال لاستخراج القيود قبل تشغيل الـ pipeline
# نشغّله على منفذ 8000 منفصل عن البقية لتجنب التعارض
import os
os.environ["OLLAMA_HOST"] = "127.0.0.1:8000"

!nohup bash -c "OLLAMA_HOST=127.0.0.1:8000 OLLAMA_ORIGIN=* ollama serve" > ollama_planner.log 2>&1 &

# انتظار بدء الخادم الكامل
!sleep 5

# فحص حالة التشغيل
!tail -20 ollama_planner.log

In [ ]:
# ─── الخلية 4: تحميل موديل Planner ──────────────────────────────────────────
# phi3.5:3.8b-mini-instruct-q4_K_M مثالي للـ Planner لأنه:
# - سريع الاستجابة (3.8B parameters فقط)
# - جيد في استخراج JSON منظّم
# - حجم صغير ≈2.3GB يتحمل بسرعة
# - نفس الموديل المستخدم في Router لتوحيد الإعدادات
ollama_model_id = "phi3.5:3.8b-mini-instruct-q4_K_M"

!OLLAMA_HOST=127.0.0.1:8000 ollama pull {ollama_model_id}

# التأكد من التحميل
!curl -s http://127.0.0.1:8000/api/tags | python3 -c "import json,sys; data=json.load(sys.stdin); [print('✓', m['name']) for m in data.get('models',[])]" 

In [ ]:
# ─── الخلية 5: اختبار استخراج القيود ────────────────────────────────────────
# نختبر قدرة الموديل على استخراج JSON منظّم من سؤال طبيعي
# هذا مثال حقيقي من الـ pipeline: سؤال عن Beverages في يونيو 1997
%%bash
curl -s http://localhost:8000/v1/chat/completions \
  -H "Content-Type: application/json" \
  -d '{
    "model": "phi3.5:3.8b-mini-instruct-q4_K_M",
    "messages": [
      {"role": "user", "content": "Extract constraints as JSON only (date_range, kpi_formula, categories, entities): What was the revenue for Beverages in June 1997?"}
    ],
    "stream": false
  }' | python3 -c "import json,sys; r=json.load(sys.stdin); print('Constraints JSON:', r['choices'][0]['message']['content'])"

In [ ]:
# ─── الخلية 6: تثبيت pyngrok ─────────────────────────────────────────────────
# مكتبة Python للتحكم في Ngrok tunnels برمجياً
!pip install pyngrok -q

In [ ]:
# ─── الخلية 7: تشغيل Ngrok Tunnel ────────────────────────────────────────────
# نقرأ NGROK_AUTH_TOKEN من Colab Secrets
# بعد التشغيل، انسخ NGROK_PLANNER_URL وضعه في ملف .env
from google.colab import userdata
from pyngrok import ngrok, conf

# قراءة token المصادقة
ngrok_auth = userdata.get('NGROK_AUTH_TOKEN')
conf.get_default().auth_token = ngrok_auth

# إغلاق tunnels قديمة
ngrok.kill()

# فتح tunnel جديد
# host_header="127.0.0.1:8000" ضروري لكي يقبل Ollama الطلبات القادمة عبر Ngrok
tunnel = ngrok.connect(
    8000,
    bind_tls=True,
    host_header="127.0.0.1:8000"
)

print("=" * 60)
print("✅ Planner Model جاهز!")
print(f"🌐 NGROK_PLANNER_URL = {tunnel.public_url}")
print("📋 انسخ هذا الرابط وضعه في ملف .env")
print("=" * 60)

In [ ]:
# ─── الخلية 8: اختبار Ngrok باستخراج قيود حقيقية ────────────────────────────
# نتحقق من أن الـ Planner يستخرج القيود بشكل صحيح عبر الرابط العام
import requests, json

ngrok_url = tunnel.public_url
model_id = ollama_model_id

test_question = "What was the profit margin for top customers in Winter 1997 holiday season?"

prompt = f"""Extract constraints as valid JSON only (no explanation).
Fields: date_range (null or {{start, end}}), kpi_formula (SQL expression or ''),
categories (list), entities (list).
Question: {test_question}"""

resp = requests.post(
    f"{ngrok_url}/v1/chat/completions",
    json={
        "model": model_id,
        "messages": [{"role": "user", "content": prompt}],
        "stream": False
    },
    timeout=30
)

print(f"Status: {resp.status_code}")
if resp.ok:
    raw = resp.json()["choices"][0]["message"]["content"]
    print(f"✅ Planner Output:\n{raw}")
    # محاولة parse الـ JSON
    try:
        import re
        json_match = re.search(r'\{.*\}', raw, re.DOTALL)
        if json_match:
            parsed = json.loads(json_match.group())
            print(f"\n✅ Parsed Constraints: {json.dumps(parsed, indent=2)}")
    except Exception as e:
        print(f"⚠ Could not parse JSON: {e}")
else:
    print(f"❌ Error: {resp.text}")